
# Semantic Bridging over Prolog Models

Este notebook executa, **somente com base nos arquivos Prolog** `entities.pl`, `triples.pl` e `rules.pl`, as três perguntas de competência:

1. **What other products the user could buy?**
2. **Which product categories should be prioritized for a discount?**
3. **Which two products would be the most suitable candidates for a bundle discount?**

O notebook foi escrito para usar **apenas predicados que existem de fato nos arquivos Prolog**.


In [1]:
# Instalação do SWI-Prolog e dependências
!apt-get update -qq
!apt-get install -y swi-prolog > /dev/null
!pip install -q pyswip pandas

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
import os
import pandas as pd
from pyswip import Prolog
from google.colab import drive


In [3]:
# Monta o Google Drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [4]:
# Ajuste este diretório se necessário
directory = "/content/drive/My Drive/KOA RecomSys/"

entities_path = os.path.join(directory, "entities.pl")
triples_path  = os.path.join(directory, "triples.pl")
rules_path    = os.path.join(directory, "rules.pl")

print("entities_path:", entities_path)
print("triples_path :", triples_path)
print("rules_path   :", rules_path)

for path in [entities_path, triples_path, rules_path]:
    print(path, "->", "OK" if os.path.exists(path) else "NOT FOUND")

if not all(os.path.exists(p) for p in [entities_path, triples_path, rules_path]):
    raise FileNotFoundError("Um ou mais arquivos Prolog não foram encontrados no diretório informado.")


entities_path: /content/drive/My Drive/KOA RecomSys/entities.pl
triples_path : /content/drive/My Drive/KOA RecomSys/triples.pl
rules_path   : /content/drive/My Drive/KOA RecomSys/rules.pl
/content/drive/My Drive/KOA RecomSys/entities.pl -> OK
/content/drive/My Drive/KOA RecomSys/triples.pl -> OK
/content/drive/My Drive/KOA RecomSys/rules.pl -> OK


In [5]:
# Carrega a base de conhecimento
prolog = Prolog()

def load_knowledge_base():
    global prolog
    prolog = Prolog()
    prolog.consult(entities_path.replace("\\", "/"))
    prolog.consult(triples_path.replace("\\", "/"))
    prolog.consult(rules_path.replace("\\", "/"))
    print("✅ Base de conhecimento carregada com sucesso.")

load_knowledge_base()


✅ Base de conhecimento carregada com sucesso.


In [6]:
# Verificação dos predicados que serão usados no notebook
predicates_to_check = [
    "product_name/2",
    "product_brand/2",
    "product_category/2",
    "same_brand/2",
    "same_category/2",
    "related_also_buy/2",
    "related_also_view/2",
    "unpack_relation/3",
    "user_intent_alignment/2",
    "semantic_recommendation/3",
    "explain_recommendation/3",
    "count_user_purchases_in_category/3",
    "count_user_also_buy_targets_in_category/3",
    "user_mentions_category/2",
    "prioritized_discount_category/4",
    "candidate_bundle_pair/2",
    "bundle_score/3",
    "bundle_explanation/3",
    "bundle_discount_candidate/6"
]

missing = []
for pred in predicates_to_check:
    exists = bool(list(prolog.query(f"current_predicate({pred})", maxresult=1)))
    print(f"{pred}: {'OK' if exists else 'MISSING'}")
    if not exists:
        missing.append(pred)

if missing:
    raise RuntimeError(f"Os seguintes predicados não existem na base carregada: {missing}")


product_name/2: OK
product_brand/2: OK
product_category/2: OK
same_brand/2: OK
same_category/2: OK
related_also_buy/2: OK
related_also_view/2: OK
unpack_relation/3: OK
user_intent_alignment/2: OK
semantic_recommendation/3: OK
explain_recommendation/3: OK
count_user_purchases_in_category/3: OK
count_user_also_buy_targets_in_category/3: OK
user_mentions_category/2: OK
prioritized_discount_category/4: OK
candidate_bundle_pair/2: OK
bundle_score/3: OK
bundle_explanation/3: OK
bundle_discount_candidate/6: OK


In [7]:
def run_query(query, limit=None):
    """Executa uma query Prolog e retorna lista de dicionários."""
    results = []
    try:
        for i, sol in enumerate(prolog.query(query)):
            results.append({k: (v.decode('utf-8') if isinstance(v, bytes) else v) for k, v in sol.items()})
            if limit is not None and i + 1 >= limit:
                break
    except Exception as e:
        raise RuntimeError(f"Erro ao executar query Prolog:\n{query}\n\nDetalhe: {e}")
    return results

def ask_bool(query):
    return len(run_query(query, limit=1)) > 0


In [8]:
# Usuário alvo
USER_ID = "U0117"

user_info = run_query(f"agent('{USER_ID}', Name, Profile)", limit=1)
if not user_info:
    raise RuntimeError(f"Usuário {USER_ID} não encontrado na base.")

print("Usuário:", user_info[0]["Name"])
print("Perfil :", user_info[0]["Profile"])


Usuário: Kylie Chambers
Perfil : Student


## CQ1 — What other products the user could buy?

In [9]:
def answer_cq1(user_id, limit=20):
    # Usa apenas predicados existentes no rules.pl atual
    query = (
        f"semantic_recommendation('{user_id}', TargetProd, Reason), "
        f"product_name(TargetProd, TargetName), "
        f"explain_recommendation('{user_id}', TargetProd, Explanation)"
    )
    raw = run_query(query)

    results = []
    seen = set()

    for row in raw:
        target_prod = row["TargetProd"]
        if target_prod in seen:
            continue
        seen.add(target_prod)

        reason = row["Reason"]
        target_name = row["TargetName"]

        if reason == "Brand Loyalty":
            evidence_query = (
                f"buys('{user_id}', SourceProd), "
                f"product_name(SourceProd, SourceName), "
                f"same_brand(SourceProd, '{target_prod}')"
            )
            evidence = run_query(evidence_query)
            evidence_type = "same_brand"
            unpacked_type = None
        else:
            evidence_query = (
                f"buys('{user_id}', SourceProd), "
                f"product_name(SourceProd, SourceName), "
                f"unpack_relation(SourceProd, '{target_prod}', UnpackedType)"
            )
            evidence = run_query(evidence_query)
            evidence_type = "unpack_relation"
            unpacked_type = sorted({ev["UnpackedType"] for ev in evidence}) if evidence else []

        intent_alignment = ask_bool(f"user_intent_alignment('{user_id}', '{target_prod}')")

        results.append({
            "target_product_id": target_prod,
            "target_product_name": target_name,
            "reason": reason,
            "intent_alignment": intent_alignment,
            "evidence_type": evidence_type,
            "unpacked_types": unpacked_type,
            "evidence_rows": evidence,
            "explanation": row["Explanation"]
        })

    return results[:limit]

cq1_results = answer_cq1(USER_ID)
print(f"Resultados CQ1: {len(cq1_results)}")


Resultados CQ1: 6


In [10]:
print("CQ1 - What other products the user could buy?")
print("=" * 90)

if not cq1_results:
    print("Nenhuma recomendação encontrada.")
else:
    for i, row in enumerate(cq1_results, 1):
        print(f"\n[{i}] Produto recomendado: {row['target_product_name']} ({row['target_product_id']})")
        print(f"    Reason              : {row['reason']}")
        print(f"    Intent alignment    : {row['intent_alignment']}")
        print(f"    Explanation         : {row['explanation']}")
        print(f"    How it was generated:")
        if row["evidence_type"] == "same_brand":
            for ev in row["evidence_rows"]:
                print(f"      - buys({USER_ID}, {ev['SourceProd']}) + same_brand({ev['SourceProd']}, {row['target_product_id']})")
                print(f"        Source product: {ev['SourceName']}")
        else:
            for ev in row["evidence_rows"]:
                print(f"      - buys({USER_ID}, {ev['SourceProd']}) + unpack_relation({ev['SourceProd']}, {row['target_product_id']}, '{ev['UnpackedType']}')")
                print(f"        Source product: {ev['SourceName']}")


CQ1 - What other products the user could buy?

[1] Produto recomendado: HP PLC Portable Laptops Model-783 (P2297)
    Reason              : Based on previous purchases, this provides Solution Composition.
    Intent alignment    : False
    Explanation         : Hello Kylie Chambers, we suggest HP PLC Portable Laptops Model-783. Reason: Based on previous purchases, this provides Solution Composition.
    How it was generated:
      - buys(U0117, P2646) + unpack_relation(P2646, P2297, 'Solution Composition')
        Source product: Sony Group High-Speed Accessories Model-890

[2] Produto recomendado: Logitech LLC Wireless Components Model-376 (P3299)
    Reason              : Based on previous purchases, this provides Solution Composition.
    Intent alignment    : False
    Explanation         : Hello Kylie Chambers, we suggest Logitech LLC Wireless Components Model-376. Reason: Based on previous purchases, this provides Solution Composition.
    How it was generated:
      - buys(U011

## CQ2 — Which product categories should be prioritized for a discount?

In [11]:
def answer_cq2(user_id, limit=20):
    query = f"prioritized_discount_category('{user_id}', Category, PriorityScore, Explanation)"
    raw = run_query(query)

    results = []
    seen = set()

    for row in raw:
        category = row["Category"]
        if category in seen:
            continue
        seen.add(category)

        mention = ask_bool(f"user_mentions_category('{user_id}', '{category}')")

        bought_count_rows = run_query(
            f"count_user_purchases_in_category('{user_id}', '{category}', Count)",
            limit=1
        )
        also_buy_count_rows = run_query(
            f"count_user_also_buy_targets_in_category('{user_id}', '{category}', Count)",
            limit=1
        )

        bought_count = bought_count_rows[0]["Count"] if bought_count_rows else 0
        also_buy_count = also_buy_count_rows[0]["Count"] if also_buy_count_rows else 0

        results.append({
            "category": category,
            "priority_score": row["PriorityScore"],
            "mentioned_by_user": mention,
            "previous_purchases_in_category": bought_count,
            "also_buy_targets_in_category": also_buy_count,
            "explanation": row["Explanation"]
        })

    results.sort(key=lambda x: x["priority_score"], reverse=True)
    return results[:limit]

cq2_results = answer_cq2(USER_ID)
print(f"Resultados CQ2: {len(cq2_results)}")


Resultados CQ2: 1


In [12]:
print("CQ2 - Which product categories should be prioritized for a discount?")
print("=" * 90)

if not cq2_results:
    print("Nenhuma categoria encontrada.")
else:
    for i, row in enumerate(cq2_results, 1):
        print(f"\n[{i}] Category: {row['category']}")
        print(f"    Priority Score                : {row['priority_score']}")
        print(f"    Mentioned by user             : {row['mentioned_by_user']}")
        print(f"    Previous purchases in category: {row['previous_purchases_in_category']}")
        print(f"    Also-buy targets in category  : {row['also_buy_targets_in_category']}")
        print(f"    Explanation                   : {row['explanation']}")
        print("    How it was generated:")
        print(f"      - user_mentions_category({USER_ID}, {row['category']}) = {row['mentioned_by_user']}")
        print(f"      - count_user_purchases_in_category({USER_ID}, {row['category']}, Count) = {row['previous_purchases_in_category']}")
        print(f"      - count_user_also_buy_targets_in_category({USER_ID}, {row['category']}, Count) = {row['also_buy_targets_in_category']}")


CQ2 - Which product categories should be prioritized for a discount?

[1] Category: C199
    Priority Score                : 10
    Mentioned by user             : True
    Previous purchases in category: 0
    Also-buy targets in category  : 0
    Explanation                   : Category C199 prioritized because mention=yes, also_buy_candidates=0, previous_purchases=0
    How it was generated:
      - user_mentions_category(U0117, C199) = True
      - count_user_purchases_in_category(U0117, C199, Count) = 0
      - count_user_also_buy_targets_in_category(U0117, C199, Count) = 0


## CQ3 — Which two products would be the most suitable candidates for a bundle discount?

In [13]:
def answer_cq3(limit=20):
    query = (
        "bundle_discount_candidate(ProductID1, ProductName1, ProductID2, ProductName2, BundleScore, Explanation)"
    )
    raw = run_query(query)

    results = []
    seen = set()

    for row in raw:
        pair_key = (row["ProductID1"], row["ProductID2"])
        if pair_key in seen:
            continue
        seen.add(pair_key)

        p1 = row["ProductID1"]
        p2 = row["ProductID2"]

        also_buy = ask_bool(f"related_also_buy('{p1}', '{p2}')")
        also_view = ask_bool(f"related_also_view('{p1}', '{p2}')")
        same_brand = ask_bool(f"same_brand('{p1}', '{p2}')")
        same_category = ask_bool(f"same_category('{p1}', '{p2}')")

        results.append({
            "product_1_id": p1,
            "product_1_name": row["ProductName1"],
            "product_2_id": p2,
            "product_2_name": row["ProductName2"],
            "bundle_score": row["BundleScore"],
            "related_also_buy": also_buy,
            "related_also_view": also_view,
            "same_brand": same_brand,
            "same_category": same_category,
            "explanation": row["Explanation"]
        })

    results.sort(key=lambda x: x["bundle_score"], reverse=True)
    return results[:limit]

cq3_results = answer_cq3(limit=20)
print(f"Resultados CQ3: {len(cq3_results)}")


Resultados CQ3: 20


In [14]:
print("CQ3 - Which two products would be the most suitable candidates for a bundle discount?")
print("=" * 90)

if not cq3_results:
    print("Nenhum par de produtos encontrado.")
else:
    for i, row in enumerate(cq3_results, 1):
        print(f"\n[{i}] Pair:")
        print(f"    Product 1     : {row['product_1_name']} ({row['product_1_id']})")
        print(f"    Product 2     : {row['product_2_name']} ({row['product_2_id']})")
        print(f"    Bundle Score  : {row['bundle_score']}")
        print(f"    Explanation   : {row['explanation']}")
        print("    How it was generated:")
        print(f"      - related_also_buy({row['product_1_id']}, {row['product_2_id']}) = {row['related_also_buy']}")
        print(f"      - related_also_view({row['product_1_id']}, {row['product_2_id']}) = {row['related_also_view']}")
        print(f"      - same_brand({row['product_1_id']}, {row['product_2_id']}) = {row['same_brand']}")
        print(f"      - same_category({row['product_1_id']}, {row['product_2_id']}) = {row['same_category']}")


CQ3 - Which two products would be the most suitable candidates for a bundle discount?

[1] Pair:
    Product 1     : Samsung LLC Ergonomic Accessories Model-211 (P1119)
    Product 2     : Dell PLC Ultrawide Storage Model-497 (P3003)
    Bundle Score  : 8
    Explanation   : Bundle candidate because also_buy=yes, also_view=yes, same_brand=no, same_category=no
    How it was generated:
      - related_also_buy(P1119, P3003) = True
      - related_also_view(P1119, P3003) = True
      - same_brand(P1119, P3003) = False
      - same_category(P1119, P3003) = False

[2] Pair:
    Product 1     : Dell PLC Ultrawide Storage Model-497 (P3003)
    Product 2     : Samsung LLC Ergonomic Accessories Model-211 (P1119)
    Bundle Score  : 8
    Explanation   : Bundle candidate because also_buy=yes, also_view=yes, same_brand=no, same_category=no
    How it was generated:
      - related_also_buy(P3003, P1119) = True
      - related_also_view(P3003, P1119) = True
      - same_brand(P3003, P1119) = Fals